In [ ]:
!pip -q install -U bitsandbytes --no-deps
!pip -q install -U accelerate peft datasets scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 24.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 10.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 5.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 21.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 39.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ---- 1) Imports & setup ----
import os, random, json
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from datasets import Dataset

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
    DataCollatorWithPadding, TrainingArguments, Trainer, set_seed, TrainerCallback
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model

# Kaggle Secrets → HF token
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HUGGINGFACE_TOKEN")
assert HF_TOKEN, "Add HUGGINGFACE_TOKEN in Kaggle Settings → Secrets."

# Env tuning
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_home")
os.environ.setdefault("HF_HUB_CACHE", "/kaggle/working/hf_home/hub")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

# Seed
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)

import transformers
print("Transformers version:", transformers.__version__)


2025-09-22 21:23:14.975059: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758576195.284759      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758576195.371234      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Transformers version: 4.52.4


In [3]:
# ---- 2) Data paths & load ----
TEXT_COL, LABEL_COL = "Review", "Label"
PREP_DIR = "/kaggle/input/fakereview/Dataset/prepared_paper_20250915_161745"  # change if needed

train_df = pd.read_csv(f"{PREP_DIR}/train.csv")
val_df   = pd.read_csv(f"{PREP_DIR}/val.csv")
test_df  = pd.read_csv(f"{PREP_DIR}/test.csv")

for df,n in [(train_df,"train"), (val_df,"val"), (test_df,"test")]:
    assert TEXT_COL in df.columns and LABEL_COL in df.columns, f"{n} missing {TEXT_COL}/{LABEL_COL}"

print(f"✅ Prepared splits loaded | train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")


✅ Prepared splits loaded | train=10004, val=1251, test=1251


In [ ]:
# ---- 3) Model & tokenizer (4-bit + LoRA)  — hardened for 401 + fork warning ----
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # quiet fork warning

MODEL_NAME = "BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct"  # gated/private? make sure your token has access
MAX_LEN = 256

# Ensure HF_TOKEN exists & is clean (you set it earlier via Kaggle Secrets)
assert HF_TOKEN and isinstance(HF_TOKEN, str), "HF_TOKEN missing"
HF_TOKEN = HF_TOKEN.strip()
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN  # also picked by libraries

# auth kwargs that work across transformers versions
AUTH = {"token": HF_TOKEN}
try:
    from transformers import AutoConfig
    _ = AutoConfig.from_pretrained(MODEL_NAME, **AUTH)
except TypeError:
    AUTH = {"use_auth_token": HF_TOKEN}  # fallback for older transformers

from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model

print("⏳ Loading model…")
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        **AUTH,
    )
except Exception as e:
    # Most common: 401 Unauthorized (token missing scope / terms not accepted / wrong account)
    print(f"❌ Could not load '{MODEL_NAME}': {e}")
    print("👉 Check: 1) token has 'Read' scope, 2) you accepted the model terms with THIS account, 3) token matches that account.")
    raise  # stop here so you notice & fix access; or swap MODEL_NAME to a public model

print("✅ Model loaded.")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True, **AUTH)
    print("✅ Loaded fast tokenizer.")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, trust_remote_code=True, **AUTH)
    print("ℹ️ Loaded slow tokenizer.")

# pad token hygiene
tokenizer.padding_side = "right"
ADDED_PAD = False
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "<pad>"})
        ADDED_PAD = True
model.config.pad_token_id = tokenizer.pad_token_id
if ADDED_PAD:
    model.resize_token_embeddings(len(tokenizer))

# 4-bit prep + LoRA
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)

# safer gradient checkpointing across versions
try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model.gradient_checkpointing_enable()

model.print_trainable_parameters()


In [ ]:
# ---- 4) Tokenization & datasets ----
def tok(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True, max_length=MAX_LEN,
        padding=False, return_attention_mask=True,
    )

train_ds = Dataset.from_pandas(train_df).map(tok, batched=True).rename_column(LABEL_COL, "labels")
val_ds   = Dataset.from_pandas(val_df).map(tok, batched=True).rename_column(LABEL_COL, "labels")
test_ds  = Dataset.from_pandas(test_df).map(tok, batched=True).rename_column(LABEL_COL, "labels")

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


In [ ]:
# ---- 5) Metrics ----
def compute_metrics(eval_pred):
    # robust to HF versions
    if isinstance(eval_pred, tuple):
        logits, labels = eval_pred
    else:
        logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}


In [ ]:
# ---- 6) HALF-EPOCH checkpoint callback (uses native Trainer saver) ----
class HalfEpochSaveCallback(TrainerCallback):
    """
    0.5, 1.0, 1.5, ... epoch cross করলেই `control.should_save = True`,
    ফলে HF Trainer পুরো checkpoint (model+optimizer+scheduler+state) সেভ করবে।
    """
    def __init__(self):
        self.last_half_index = -1  # tracks int(epoch * 2): 0,1,2,...

    def on_step_end(self, args, state, control, **kwargs):
        if state.epoch is None:
            return control
        # epsilon to avoid float drift (e.g., 0.499999)
        half_index = int((state.epoch + 1e-9) * 2)
        if half_index > self.last_half_index:
            self.last_half_index = half_index
            epoch_float = half_index / 2.0
            control.should_save = True   # ✅ trigger full checkpoint save
            if state.is_local_process_zero:
                print(f"💾 Half-epoch {epoch_float:.1f} reached → saving checkpoint… (global_step={state.global_step})")
        return control

In [ ]:
# ---- 7) TrainingArguments ----
OUTPUT_DIR = "/kaggle/working/fake_review_ckpts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,

    num_train_epochs=3,
    learning_rate=2e-4,
    max_grad_norm=1.0,

    fp16=True,   # 4-bit + fp16 compute
    bf16=False,

    # Keep a "steps" strategy so the internal save hook is active across HF versions.
    # Use a huge save_steps so it won't auto-save except when our callback forces it.
    save_strategy="steps",
    save_steps=999999999,
    save_total_limit=3,             # keep last 3 checkpoints (older pruned)
    evaluation_strategy="epoch",    # eval each epoch end
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
)


In [ ]:
# ---- 8) Trainer ----
model.config.id2label = {0: "Fake", 1: "Non-Fake"}
model.config.label2id = {"Fake": 0, "Non-Fake": 1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,            # ✅ correct argument
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Add half-epoch checkpoint saver
trainer.add_callback(HalfEpochSaveCallback())

# (Optional) also ensure eval runs exactly at epoch end even if something changes
class EvalAtEpochEnd(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        control.should_evaluate = True
        return control
trainer.add_callback(EvalAtEpochEnd())

# ---- 9) Auto-resume (current run dir or external read-only dir via CKPT_INPUT_DIR) ----
CKPT_INPUT_DIR = os.environ.get("CKPT_INPUT_DIR", "").strip()  # optional
last_ckpt = None
try:
    last_ckpt = get_last_checkpoint(OUTPUT_DIR)
except Exception:
    last_ckpt = None

if (last_ckpt is None) and CKPT_INPUT_DIR and os.path.isdir(CKPT_INPUT_DIR):
    try:
        last_ckpt = get_last_checkpoint(CKPT_INPUT_DIR)
    except Exception:
        last_ckpt = None

if last_ckpt:
    print(f"🔁 Resuming from: {last_ckpt}")
    train_kwargs = dict(resume_from_checkpoint=last_ckpt)
else:
    print("🆕 No checkpoint found, training from scratch.")
    train_kwargs = {}


In [ ]:
# ---- 10) Train ----
trainer.train(**train_kwargs)

In [ ]:
# ---- 11) Save final model ----
final_dir = "/kaggle/working/final"
os.makedirs(final_dir, exist_ok=True)
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"✅ Final model saved to: {os.path.abspath(final_dir)}")


In [ ]:

# ---- 12) Evaluate (Validation + Test) & save artifacts ----
# Validation (final)
val_eval = trainer.evaluate(eval_dataset=val_ds)
print("📊 Validation:", val_eval)

# Test
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_pred = np.argmax(logits, axis=1)
y_true = test_df[LABEL_COL].to_numpy(dtype=int)
probs  = F.softmax(torch.tensor(logits), dim=1).cpu().numpy()

pred_df = pd.DataFrame({
    "Review": test_df[TEXT_COL].tolist(),
    "True": y_true,
    "Pred": y_pred,
    "Prob_Fake(0)": probs[:,0],
    "Prob_NonFake(1)": probs[:,1],
})
pred_df.to_csv("/kaggle/working/test_predictions_with_probs.csv", index=False, encoding="utf-8")

metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[0]),
    "recall_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[1]),
    "f1_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[2]),
}
with open("/kaggle/working/metrics_test.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

report_str = classification_report(
    y_true, y_pred, labels=[0,1],
    target_names=["Fake(0)", "Non-Fake(1)"], digits=4, zero_division=0
)
print("\n🔎 Test Classification Report:\n", report_str)
with open("/kaggle/working/classification_report_test.txt", "w", encoding="utf-8") as f:
    f.write(report_str)

cm = confusion_matrix(y_true, y_pred, labels=[0,1])
pd.DataFrame(cm, index=["True_Fake(0)","True_NonFake(1)"], columns=["Pred_Fake(0)","Pred_NonFake(1)"]).to_csv("/kaggle/working/confusion_matrix_test.csv", encoding="utf-8")

mis_df = pred_df[pred_df["True"] != pred_df["Pred"]]
mis_df.to_csv("/kaggle/working/misclassified_cases_test.csv", index=False, encoding="utf-8")

print("✅ Saved artifacts in /kaggle/working : final/, test_predictions_with_probs.csv, metrics_test.json, classification_report_test.txt, confusion_matrix_test.csv, misclassified_cases_test.csv")